In [1]:
# ============================================================
# AMP Challenge — Step 7B
# APEX-Informed Soft Preselection: 120K -> 60K
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files

TARGET_N = 60_000

# Soft ranking weights
W_POTENCY = 0.50
W_PHYSCHEM = 0.30
W_NOVELTY = 0.20

# Optional soft penalty for candidates already flagged >=80%
HIGH_SIM_PENALTY = 0.10

print("Upload STEP7_120K_WITH_APEX_POTENCY.csv")
uploaded = files.upload()

input_file = next(iter(uploaded))
print("Input:", input_file)

df = pd.read_csv(input_file, low_memory=False)

# ============================================================
# 1. Basic validation
# ============================================================

required = [
    "sequence",
    "batch",
    "apex_mean_mic_uM",
    "APD6_physchem_realism_diagnostic",
    "external_novelty_diagnostic",
    "high_similarity_flag_ge80",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["sequence"] = (
    df["sequence"]
    .astype(str)
    .str.strip()
    .str.upper()
)

if len(df) != 120_000:
    raise ValueError(
        f"Expected 120,000 rows, found {len(df):,}"
    )

if df["sequence"].duplicated().any():
    raise ValueError(
        f"Duplicate sequences found: "
        f"{df['sequence'].duplicated().sum()}"
    )

for col in [
    "apex_mean_mic_uM",
    "APD6_physchem_realism_diagnostic",
    "external_novelty_diagnostic",
]:
    if df[col].isna().any():
        raise ValueError(
            f"Missing values detected in {col}: "
            f"{df[col].isna().sum()}"
        )

print("Input validation passed.")
print("Rows:", len(df))
print("Unique sequences:", df["sequence"].nunique())


# ============================================================
# 2. Normalize potency
# ============================================================
# Lower MIC = better.
#
# Robust percentile ranking is used instead of min-max scaling,
# because it avoids excessive influence from extreme MIC values.
#
# Best candidate approaches 1.0.
# Worst candidate approaches 0.0.
# ============================================================

df["score_apex_potency"] = (
    1.0
    - df["apex_mean_mic_uM"]
      .rank(method="average", pct=True)
)

# ============================================================
# 3. Existing normalized signals
# ============================================================

df["score_physchem_7B"] = (
    df["APD6_physchem_realism_diagnostic"]
    .clip(0, 100)
    / 100.0
)

df["score_novelty_7B"] = (
    df["external_novelty_diagnostic"]
    .clip(0, 100)
    / 100.0
)

# ============================================================
# 4. Composite soft preselection score
# ============================================================

df["step7b_preselection_score"] = (
    W_POTENCY * df["score_apex_potency"]
    + W_PHYSCHEM * df["score_physchem_7B"]
    + W_NOVELTY * df["score_novelty_7B"]
)

# Soft penalty only; NOT a hard rejection.
flag = (
    df["high_similarity_flag_ge80"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)

df.loc[
    flag,
    "step7b_preselection_score"
] -= HIGH_SIM_PENALTY


# ============================================================
# 5. Balanced batch-aware selection
# ============================================================
# Preserve representation from all five HydrAMP generation
# batches. 60,000 / 5 = 12,000 candidates per batch.
#
# If a batch unexpectedly lacks enough candidates, remaining
# slots are globally refilled from the best unselected rows.
# ============================================================

batches = sorted(df["batch"].dropna().unique())

if len(batches) != 5:
    raise ValueError(
        f"Expected 5 batches, found {len(batches)}: {batches}"
    )

BASE_QUOTA = TARGET_N // len(batches)

selected_indices = []

for batch in batches:

    sub = df[df["batch"] == batch].copy()

    sub = sub.sort_values(
        [
            "step7b_preselection_score",
            "apex_mean_mic_uM",
            "external_max_similarity_pct",
        ],
        ascending=[False, True, True],
        kind="mergesort",
    )

    take_n = min(BASE_QUOTA, len(sub))

    selected_indices.extend(
        sub.head(take_n).index.tolist()
    )

selected_indices = list(dict.fromkeys(selected_indices))

# ============================================================
# 6. Global refill if required
# ============================================================

if len(selected_indices) < TARGET_N:

    remaining = df.loc[
        ~df.index.isin(selected_indices)
    ].copy()

    remaining = remaining.sort_values(
        [
            "step7b_preselection_score",
            "apex_mean_mic_uM",
            "external_max_similarity_pct",
        ],
        ascending=[False, True, True],
        kind="mergesort",
    )

    need = TARGET_N - len(selected_indices)

    selected_indices.extend(
        remaining.head(need).index.tolist()
    )


# ============================================================
# 7. Construct final 60K pool
# ============================================================

out = df.loc[selected_indices].copy()

out = out.sort_values(
    [
        "step7b_preselection_score",
        "apex_mean_mic_uM",
        "external_max_similarity_pct",
    ],
    ascending=[False, True, True],
    kind="mergesort",
).reset_index(drop=True)

out["step7b_rank"] = np.arange(
    1,
    len(out) + 1
)

# ============================================================
# 8. Final validation
# ============================================================

if len(out) != TARGET_N:
    raise RuntimeError(
        f"Expected {TARGET_N:,}, obtained {len(out):,}"
    )

if out["sequence"].duplicated().any():
    raise RuntimeError(
        "Duplicate sequence survived final selection."
    )

if out["step7b_preselection_score"].isna().any():
    raise RuntimeError(
        "Missing Step7B score detected."
    )

# ============================================================
# 9. Summary
# ============================================================

print("\n" + "=" * 65)
print("STEP 7B COMPLETE")
print("=" * 65)

print(f"Input candidates:  {len(df):,}")
print(f"Selected:          {len(out):,}")
print(
    f"Unique sequences:  "
    f"{out['sequence'].nunique():,}"
)
print(
    f"Duplicates:        "
    f"{out['sequence'].duplicated().sum()}"
)

print("\nCandidates per batch:")
print(out["batch"].value_counts().sort_index())

print("\nStep7B score:")
print(
    out["step7b_preselection_score"]
    .describe(
        percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
    )
)

print("\nAPEX mean MIC in selected 60K:")
print(
    out["apex_mean_mic_uM"]
    .describe(
        percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
    )
)

print(
    "\nSelected with external similarity >=80%:",
    int(
        out["high_similarity_flag_ge80"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
        .sum()
    )
)

# ============================================================
# 10. Save
# ============================================================

OUTPUT = "STEP7B_PRESELECTED_60K.csv"

out.to_csv(
    OUTPUT,
    index=False
)

print("\nSaved:", OUTPUT)

files.download(OUTPUT)

Upload STEP7_120K_WITH_APEX_POTENCY.csv


Saving STEP7_120K_WITH_APEX_POTENCY.csv to STEP7_120K_WITH_APEX_POTENCY.csv
Input: STEP7_120K_WITH_APEX_POTENCY.csv
Input validation passed.
Rows: 120000
Unique sequences: 120000

STEP 7B COMPLETE
Input candidates:  120,000
Selected:          60,000
Unique sequences:  60,000
Duplicates:        0

Candidates per batch:
batch
hydramp_batch1_seed42    12000
hydramp_batch2_seed44    12000
hydramp_batch3_seed46    12000
hydramp_batch4_seed48    12000
hydramp_batch5_seed50    12000
Name: count, dtype: int64

Step7B score:
count    60000.000000
mean         0.719230
std          0.070482
min          0.596016
5%           0.609482
25%          0.658229
50%          0.719072
75%          0.779717
95%          0.827485
max          0.885153
Name: step7b_preselection_score, dtype: float64

APEX mean MIC in selected 60K:
count    60000.000000
mean       180.677099
std         57.142245
min         33.019216
5%          85.167175
25%        135.773656
50%        183.226502
75%        227.819332
95

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>